### **Silver Notebook**

In [1]:
from pyspark.sql.functions import *
from pyspark.sql. types import *

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 3, Finished, Available, Finished, False)

**Utilities**

# **Data Reading**

**Same Lakehouse**

In [2]:
df = spark.read.format("csv")\
.option("header", True)\
.option("inferSchema", True)\
.load("Files/StoreDataShrcutt")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 4, Finished, Available, Finished, False)

In [3]:
display(df)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d86548c2-ef80-4499-85ff-39777671e09f)

## **Bronze Lakehouse**

**Customers Data**

In [4]:
df_cust = spark.read.format("delta")\
          .load("abfss://Akash_WS@onelake.dfs.fabric.microsoft.com/BronzeLH.Lakehouse/Tables/dbo/customer_dim")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 6, Finished, Available, Finished, False)

In [5]:
display(df_cust)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 992b34b0-6d64-48b9-936e-a1d798dcd0b9)

In [6]:
df_cust = spark.read.table("BronzeLH.dbo.customer_dim")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 8, Finished, Available, Finished, False)

In [7]:
display(df_cust)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45ac358a-2178-4dd3-8186-1a83f977bb2c)

**Transfromation**

In [8]:
df_cust = df_cust.withColumn("firstname", split(col('name'), ' ')[0])\
                .withColumn("Lastname", split(col('name'), ' ')[1])


display(df_cust)                



StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0a55905c-1681-4398-ae59-5f5e51da1ea0)

In [9]:
df_cust = df_cust.fillna({"lastname": "NA"})
display(df_cust)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 519c916f-6faa-472f-b48b-177b44c4b63b)

In [11]:
df_cust.write.format("delta")\
              .mode("overwrite")\
              .option("path","abfss://Akash_WS@onelake.dfs.fabric.microsoft.com/SilverLH.Lakehouse/Files/ExtData")\
              .saveAsTable("SilverLH.dbo.ExtData")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 13, Finished, Available, Finished, False)

In [13]:
df_cust.write.format("delta")\
              .mode("overwrite")\
              .option("path","abfss://Akash_WS@onelake.dfs.fabric.microsoft.com/BronzeLH.Lakehouse/Files/ExtData2")\
              .saveAsTable("BronzeLH.dbo.ExtData2")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 15, Finished, Available, Finished, False)

**Fact table**

In [14]:
df_fact = spark.read.table("BronzeLH.dbo.fact_table")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 16, Finished, Available, Finished, False)

In [15]:
display(df_fact)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fef4210a-bb20-4202-837b-1e99e9b9dad1)

In [16]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType

df_fact = (
    df_fact
    .withColumn("quantity", col("quantity").cast(IntegerType()))
    .withColumn("unit_price", col("unit_price").cast(DoubleType()))
    .withColumn("total_price", col("total_price").cast(DoubleType()))
)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 18, Finished, Available, Finished, False)

In [17]:
df_fact.printSchema()

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 19, Finished, Available, Finished, False)

root
 |-- payment_key: string (nullable = true)
 |-- coustomer_key: string (nullable = true)
 |-- time_key: string (nullable = true)
 |-- item_key: string (nullable = true)
 |-- store_key: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_price: double (nullable = true)



In [18]:
display(df_fact)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dec3fa4f-f716-435f-bf93-13de4b84c664)

In [19]:
df_fact.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("SilverLH.dbo.fact_table")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 21, Finished, Available, Finished, False)

**Store dim**

In [20]:
df_store = spark.read.table("BronzeLH.dbo.store_dim")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 22, Finished, Available, Finished, False)

In [21]:
display(df_store)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 12cc4505-720f-4803-810f-8278faa0a322)

In [22]:
df_store = df.withColumn("address", concat(col("district"), lit("-"), col("upazila")))
df_store = df_store.drop("district","upazila")
display(df_store)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a2c4691c-5c45-4be9-91d4-41a55919601e)

In [23]:
df_store.write.format("delta")\
              .mode("overwrite")\
              .saveAsTable("SilverLH.dbo.store_dim")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 25, Finished, Available, Finished, False)

**Trans dim**

In [24]:
df_trans = spark.read.table("BronzeLH.dbo.trans_dim")
display(df_trans)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6803d677-b58b-4ed9-ba97-82229cb7b5ef)

In [25]:
df_trans = df_trans.withColumn(
    "bank_name",
    regexp_replace(col("bank_name"), "None", "Not Available")
)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 27, Finished, Available, Finished, False)

In [26]:
display(df_trans)

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f248c71-8201-4ea4-b036-444c13cbee53)

In [27]:
df_trans.write.format("delta")\
              .mode("overwrite")\
              .saveAsTable("SilverLH.dbo.trans_dim")

StatementMeta(, 6a03705e-906d-4e7f-8aef-f7195e00a20b, 29, Finished, Available, Finished, True)